In [2]:
import pandas as pd

In [3]:
poi_path = '/home/z5562390/work/dataset2023/cell_POIcat.csv.gz'
cat_path = '/home/z5562390/work/dataset2023/POI_datacategories.csv'

poi_df = pd.read_csv(poi_path)
cat_df = pd.read_csv(cat_path, header=None, names=['category_name'])

print("POI data shape:", poi_df.shape)
print("Category mapping shape:", cat_df.shape)
print()
print(poi_df.head())
print()
print(cat_df.head(10))

POI data shape: (221159, 4)
Category mapping shape: (85, 1)

   x  y  POIcategory  POI_count
0  1  1           48          4
1  1  1           58          1
2  1  1           59          1
3  1  1           69          2
4  1  1           73          1

                category_name
0                        Food
1                    Shopping
2               Entertainment
3         Japanese restaurant
4          Western restaurant
5  Eat all you can restaurant
6          Chinese restaurant
7           Indian restaurant
8            Ramen restaurant
9            Curry restaurant


In [ ]:
cat_df['POIcategory'] = cat_df.index + 1

poi_named_df = poi_df.merge(cat_df, on='POIcategory', how='left')

print("Merged POI data:")
print(poi_named_df.head(10))

Merged POI data:
   x  y  POIcategory  POI_count      category_name
0  1  1           48          4    Transit Station
1  1  1           58          1       Kindergarten
2  1  1           59          1        Real Estate
3  1  1           69          2         Hair Salon
4  1  1           73          1   Community Center
5  1  1           74          4             Church
6  1  1           79          2  Building Material
7  1  2           48          1    Transit Station
8  1  2           60          1    Home Appliances
9  1  2           61          2        Post Office


In [ ]:
check_ids = [48, 58, 59, 69, 73]

print(cat_df[cat_df['POIcategory'].isin(check_ids)])

       category_name  POIcategory
47   Transit Station           48
57      Kindergarten           58
58       Real Estate           59
68        Hair Salon           69
72  Community Center           73


In [6]:
def build_grid_summary(group, top_k=8):
    group = group.sort_values('POI_count', ascending=False)
    parts = []

    for _, row in group.head(top_k).iterrows():
        cat_name = row['category_name']
        count = int(row['POI_count'])
        parts.append(f"{cat_name}({count})")

    return ", ".join(parts)

In [7]:
grid_summary_df = (
    poi_named_df.groupby(['x', 'y'])
    .apply(build_grid_summary)
    .reset_index(name='poi_summary')
)

print("Grid summary preview:")
print(grid_summary_df.head(10))
print()
print("Number of unique grids:", len(grid_summary_df))

Grid summary preview:
   x   y                                        poi_summary
0  1   1  Transit Station(4), Church(4), Hair Salon(2), ...
1  1   2  Post Office(2), Transit Station(1), Home Appli...
2  1   3  Church(4), Transit Station(2), NPO(2), Heavy I...
3  1   4  Community Center(3), Heavy Industry(2), Transi...
4  1   5  Home Appliances(2), Church(2), Transit Station...
5  1   6  Transit Station(5), Community Center(3), Heavy...
6  1   7  Transit Station(4), Building Material(3), Elde...
7  1   8  Transit Station(3), Western restaurant(2), Chu...
8  1   9  Transit Station(7), Bank(5), Hospital(4), Chur...
9  1  10  NPO(5), Community Center(5), Park(5), Transit ...

Number of unique grids: 20146


/tmp/ipykernel_147866/2269308737.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_grid_summary)


In [8]:
def classify_grid(summary_text):
    text = summary_text.lower()

    food_keywords = [
        'restaurant', 'cafe', 'bakery', 'bar', 'pizza',
        'sweets', 'diner', 'cuisine', 'tea salon'
    ]
    shopping_keywords = ['shopping', 'retail']
    leisure_keywords = ['entertainment']
    
    food_score = sum(k in text for k in food_keywords)
    shopping_score = sum(k in text for k in shopping_keywords)
    leisure_score = sum(k in text for k in leisure_keywords)

    if food_score >= 2 and shopping_score >= 1:
        return 'mixed_food_commercial'
    elif food_score >= 2:
        return 'food_leisure'
    elif shopping_score >= 1:
        return 'commercial'
    elif leisure_score >= 1:
        return 'entertainment'
    else:
        return 'other'

In [9]:
grid_summary_df['semantic_label'] = grid_summary_df['poi_summary'].apply(classify_grid)

print(grid_summary_df.head(15))

    x   y                                        poi_summary  \
0   1   1  Transit Station(4), Church(4), Hair Salon(2), ...   
1   1   2  Post Office(2), Transit Station(1), Home Appli...   
2   1   3  Church(4), Transit Station(2), NPO(2), Heavy I...   
3   1   4  Community Center(3), Heavy Industry(2), Transi...   
4   1   5  Home Appliances(2), Church(2), Transit Station...   
5   1   6  Transit Station(5), Community Center(3), Heavy...   
6   1   7  Transit Station(4), Building Material(3), Elde...   
7   1   8  Transit Station(3), Western restaurant(2), Chu...   
8   1   9  Transit Station(7), Bank(5), Hospital(4), Chur...   
9   1  10  NPO(5), Community Center(5), Park(5), Transit ...   
10  1  11  Transit Station(4), Japanese restaurant(3), He...   
11  1  12  Heavy Industry(3), Real Estate(1), Church(1), ...   
12  1  13  Hospital(1), Laundry (1), Driving School(1), H...   
13  1  14  Western restaurant(1), Elderly Care Home(1), C...   
14  1  16  Laundry (1), Retail Store(1),

In [10]:
def classify_grid(summary_text):
    text = summary_text.lower()

    food_keywords = [
        'restaurant', 'cafe', 'bakery', 'bar', 'pizza',
        'sweets', 'diner', 'cuisine', 'tea salon'
    ]
    shopping_keywords = ['shopping', 'retail']
    leisure_keywords = ['entertainment']
    
    food_score = sum(k in text for k in food_keywords)
    shopping_score = sum(k in text for k in shopping_keywords)
    leisure_score = sum(k in text for k in leisure_keywords)

    if food_score >= 2 and shopping_score >= 1:
        return 'mixed_food_commercial'
    elif food_score >= 2:
        return 'food_leisure'
    elif shopping_score >= 1:
        return 'commercial'
    elif leisure_score >= 1:
        return 'entertainment'
    else:
        return 'other'

In [11]:
grid_summary_df['semantic_label'] = grid_summary_df['poi_summary'].apply(classify_grid)

print(grid_summary_df.head(15))

    x   y                                        poi_summary  \
0   1   1  Transit Station(4), Church(4), Hair Salon(2), ...   
1   1   2  Post Office(2), Transit Station(1), Home Appli...   
2   1   3  Church(4), Transit Station(2), NPO(2), Heavy I...   
3   1   4  Community Center(3), Heavy Industry(2), Transi...   
4   1   5  Home Appliances(2), Church(2), Transit Station...   
5   1   6  Transit Station(5), Community Center(3), Heavy...   
6   1   7  Transit Station(4), Building Material(3), Elde...   
7   1   8  Transit Station(3), Western restaurant(2), Chu...   
8   1   9  Transit Station(7), Bank(5), Hospital(4), Chur...   
9   1  10  NPO(5), Community Center(5), Park(5), Transit ...   
10  1  11  Transit Station(4), Japanese restaurant(3), He...   
11  1  12  Heavy Industry(3), Real Estate(1), Church(1), ...   
12  1  13  Hospital(1), Laundry (1), Driving School(1), H...   
13  1  14  Western restaurant(1), Elderly Care Home(1), C...   
14  1  16  Laundry (1), Retail Store(1),

In [12]:
for _, row in grid_summary_df.head(5).iterrows():
    print(f"Grid ({row['x']}, {row['y']})")
    print("POI summary:", row['poi_summary'])
    print("Semantic label:", row['semantic_label'])
    print("-" * 60)

Grid (1, 1)
POI summary: Transit Station(4), Church(4), Hair Salon(2), Building Material(2), Kindergarten(1), Real Estate(1), Community Center(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 2)
POI summary: Post Office(2), Transit Station(1), Home Appliances(1), Laundry (1), Gardening(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 3)
POI summary: Church(4), Transit Station(2), NPO(2), Heavy Industry(2), Convenience Store(1), Elderly Care Home(1), Home Appliances(1), Kindergarten(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 4)
POI summary: Community Center(3), Heavy Industry(2), Transit Station(1), Driving School(1), Home Appliances(1), Hair Salon(1), Church(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 5)
POI summary: Home Appliances(2), Church(2), Transit Station(1), Laundry (1), Hair Salon(1),

In [13]:
demo_x = 1
demo_y = 1

demo_grid_df = poi_named_df[(poi_named_df['x'] == demo_x) & (poi_named_df['y'] == demo_y)]
demo_grid_df = demo_grid_df.sort_values('POI_count', ascending=False)

print(demo_grid_df)

   x  y  POIcategory  POI_count      category_name
0  1  1           48          4    Transit Station
5  1  1           74          4             Church
3  1  1           69          2         Hair Salon
6  1  1           79          2  Building Material
1  1  1           58          1       Kindergarten
2  1  1           59          1        Real Estate
4  1  1           73          1   Community Center


In [ ]:
# 85 -> 11（hardcode）
top11_mapping = {
    "Food": "Dining and Drinking",
    "Shopping": "Retail",
    "Entertainment": "Arts and Entertainment",
    "Japanese restaurant": "Dining and Drinking",
    "Western restaurant": "Dining and Drinking",
    "Eat all you can restaurant": "Dining and Drinking",
    "Chinese restaurant": "Dining and Drinking",
    "Indian restaurant": "Dining and Drinking",
    "Ramen restaurant": "Dining and Drinking",
    "Curry restaurant": "Dining and Drinking",
    "BBQ restaurant": "Dining and Drinking",
    "Hot pot restaurant": "Dining and Drinking",
    "Bar": "Nightlife Spot",
    "Diner": "Dining and Drinking",
    "Creative cuisine": "Dining and Drinking",
    "Organic cuisine": "Dining and Drinking",
    "Pizza": "Dining and Drinking",
    "Café": "Dining and Drinking",
    "Tea Salon": "Dining and Drinking",
    "Bakery": "Dining and Drinking",
    "Sweets ": "Dining and Drinking",
    "Wine Bar": "Nightlife Spot",
    "Pub": "Nightlife Spot",
    "Disco": "Nightlife Spot",
    "Beer Garden": "Nightlife Spot",
    "Fast Food": "Dining and Drinking",
    "Karaoke": "Arts and Entertainment",
    "Cruising": "Travel and Transportation",
    "Theme Park Restaurant": "Dining and Drinking",
    "Amusement Restaurant": "Dining and Drinking",
    "Other Restaurants": "Dining and Drinking",
    "Glasses": "Retail",
    "Drug Store": "Retail",
    "Electronics Store": "Retail",
    "DIY Store": "Retail",
    "Convenience Store": "Retail",
    "Recycle Shop": "Retail",
    "Interior Shop": "Retail",
    "Sports Store": "Retail",
    "Clothes Store": "Retail",
    "Grocery Store": "Retail",
    "Online Grocery Store": "Retail",
    "Sports Recreation": "Sports and Recreation",
    "Game Arcade": "Arts and Entertainment",
    "Swimming Pool": "Sports and Recreation",
    "Hotel": "Travel and Transportation",
    "Park": "Landmarks and Outdoors",
    "Transit Station": "Travel and Transportation",
    "Parking Area": "Travel and Transportation",
    "Casino": "Arts and Entertainment",
    "Hospital": "Health and Medicine",
    "Pharmacy": "Health and Medicine",
    "Chiropractic": "Health and Medicine",
    "Elderly Care Home": "Health and Medicine",
    "Fishing": "Sports and Recreation",
    "School": "Community and Government",
    "Cram School": "Community and Government",
    "Kindergarten": "Community and Government",
    "Real Estate": "Business and Professional Services",
    "Home Appliances": "Retail",
    "Post Office": "Community and Government",
    "Laundry ": "Business and Professional Services",
    "Driving School": "Community and Government",
    "Wedding Ceremony": "Event",
    "Cemetary": "Community and Government",
    "Bank": "Business and Professional Services",
    "Vet": "Health and Medicine",
    "Hot Spring": "Landmarks and Outdoors",
    "Hair Salon": "Business and Professional Services",
    "Lawyer Office": "Business and Professional Services",
    "Recruitment Office": "Business and Professional Services",
    "City Hall": "Community and Government",
    "Community Center": "Community and Government",
    "Church": "Community and Government",
    "Retail Store": "Retail",
    "Accountant Office": "Business and Professional Services",
    "IT Office": "Business and Professional Services",
    "Publisher Office": "Business and Professional Services",
    "Building Material": "Retail",
    "Gardening": "Business and Professional Services",
    "Heavy Industry": "Business and Professional Services",
    "NPO": "Community and Government",
    "Utility Copany": "Community and Government",
    "Port": "Travel and Transportation",
    "Research Facility": "Business and Professional Services",
}

cat_df["top11_category"] = cat_df["category_name"].map(top11_mapping)

print("Unmapped categories:")
print(cat_df[cat_df["top11_category"].isna()])

poi_named_df = poi_named_df.merge(
    cat_df[["POIcategory", "top11_category"]],
    on="POIcategory",
    how="left"
)

print("POI data with new top11 category:")
print(poi_named_df.head(10))


Unmapped categories:
Empty DataFrame
Columns: [category_name, POIcategory, top11_category]
Index: []
POI data with new top11 category:
   x  y  POIcategory  POI_count      category_name  \
0  1  1           48          4    Transit Station   
1  1  1           58          1       Kindergarten   
2  1  1           59          1        Real Estate   
3  1  1           69          2         Hair Salon   
4  1  1           73          1   Community Center   
5  1  1           74          4             Church   
6  1  1           79          2  Building Material   
7  1  2           48          1    Transit Station   
8  1  2           60          1    Home Appliances   
9  1  2           61          2        Post Office   

                       top11_category  
0           Travel and Transportation  
1            Community and Government  
2  Business and Professional Services  
3  Business and Professional Services  
4            Community and Government  
5            Community and Gov

In [ ]:
grid_top11_df = (
    poi_named_df.pivot_table(
        index=["x", "y"],
        columns="top11_category",
        values="POI_count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

print("Grid-level 11-category feature preview:")
print(grid_top11_df.head())
print("Shape:", grid_top11_df.shape)


Grid-level 11-category feature preview:
top11_category  x  y  Arts and Entertainment  \
0               1  1                       0   
1               1  2                       0   
2               1  3                       0   
3               1  4                       0   
4               1  5                       0   

top11_category  Business and Professional Services  Community and Government  \
0                                                3                         6   
1                                                2                         2   
2                                                2                         9   
3                                                3                         5   
4                                                3                         2   

top11_category  Dining and Drinking  Event  Health and Medicine  \
0                                 0      0                    0   
1                                 0      0              

In [ ]:
def build_top11_summary(group, top_k=5):
    group = (
        group.groupby("top11_category", as_index=False)["POI_count"]
        .sum()
        .sort_values("POI_count", ascending=False)
    )
    parts = []
    for _, row in group.head(top_k).iterrows():
        parts.append(f'{row["top11_category"]}({int(row["POI_count"])})')
    return ", ".join(parts)

grid_top11_summary_df = (
    poi_named_df.groupby(["x", "y"])
    .apply(build_top11_summary)
    .reset_index(name="top11_summary")
)

grid_summary_df = grid_summary_df.merge(grid_top11_summary_df, on=["x", "y"], how="left")

print("Grid summary with new top11 category layer:")
print(grid_summary_df[["x", "y", "poi_summary", "top11_summary", "semantic_label"]].head(10))


Grid summary with new top11 category layer:
   x   y                                        poi_summary  \
0  1   1  Transit Station(4), Church(4), Hair Salon(2), ...   
1  1   2  Post Office(2), Transit Station(1), Home Appli...   
2  1   3  Church(4), Transit Station(2), NPO(2), Heavy I...   
3  1   4  Community Center(3), Heavy Industry(2), Transi...   
4  1   5  Home Appliances(2), Church(2), Transit Station...   
5  1   6  Transit Station(5), Community Center(3), Heavy...   
6  1   7  Transit Station(4), Building Material(3), Elde...   
7  1   8  Transit Station(3), Western restaurant(2), Chu...   
8  1   9  Transit Station(7), Bank(5), Hospital(4), Chur...   
9  1  10  NPO(5), Community Center(5), Park(5), Transit ...   

                                       top11_summary semantic_label  
0  Community and Government(6), Travel and Transp...          other  
1  Business and Professional Services(2), Communi...          other  
2  Community and Government(9), Business and Prof...

/tmp/ipykernel_147866/2374357667.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_top11_summary)


In [ ]:
demo_x = 1
demo_y = 1

demo_grid_top11 = (
    poi_named_df[(poi_named_df["x"] == demo_x) & (poi_named_df["y"] == demo_y)]
    .groupby("top11_category", as_index=False)["POI_count"]
    .sum()
    .sort_values("POI_count", ascending=False)
)

print(f"Top11 category distribution for Grid ({demo_x}, {demo_y}):")
print(demo_grid_top11)


Top11 category distribution for Grid (1, 1):
                       top11_category  POI_count
1            Community and Government          6
3           Travel and Transportation          4
0  Business and Professional Services          3
2                              Retail          2


In [ ]:
output_path_top11 = '/home/z5562390/work/dataset2023/POI_datacategories_with_top11.csv'
grid_summary_df.to_csv(output_path_top11, index=False)

print("Saved demo with top11 summary to:", output_path_top11)


Saved demo with top11 summary to: /home/z5562390/work/dataset2023/POI_datacategories.csvwith_top11.csv
